In [ ]:
Сделал новый ноутбук,хотя домашка по линейной регрессии лежит на гитхабе.
    

In [14]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, r2_score
import time
df = pd.read_csv('ToyotaCorolla.csv')
df = pd.get_dummies(df, columns=['FuelType'], drop_first=True)

X = df.drop('Price', axis=1)
y = df['Price']


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Размер обучающей выборки: {X_train.shape}")
print(f"Размер тестовой выборки: {X_test.shape}")

Размер обучающей выборки: (1148, 10)
Размер тестовой выборки: (288, 10)


Разделение выборки
1. Разделение выборки Так как деревья решений устойчивы к разному масштабу данных, предобработка в виде масштабирования признаков не является обязательной,но я просто просмотрел наличие пропущенных значений и отобрал наиболее важные признаки.Ну и для сравнения с линейное регрессией использовать те же данные.
2. Разделил выборку как обычно, 20% для теста и 80% процентов для обучения с помощью train_test_split.
3. Обычно на 5 или 10 фолдов.
4. Не использовать кросс-валидацию можно, тогда модель будет обучаться на train/testю.
       


In [19]:
depth = 10

rf = RandomForestRegressor(n_estimators=50, max_depth=depth, n_jobs=-1, random_state=42)
t0 = time.time()
rf.fit(X_tr, y_tr)
rf_time = time.time() - t0

dt = DecisionTreeRegressor(max_depth=depth, random_state=42)
t1 = time.time()
dt.fit(X_tr, y_tr)
dt_time = time.time() - t1

def check_quality(model, name, time_spent):
    pred = model.predict(X_te)
    mae = mean_absolute_error(y_te, pred)
    rmse = np.sqrt(mean_squared_error(y_te, pred))
    r2 = r2_score(y_te, pred)
    print(f"--- {name} ---")
    print(f"MAE: {mae:.2f} | RMSE: {rmse:.2f} | R2: {r2:.4f} | Время: {time_spent:.4f}s")

check_quality(rf, "Случайный лес", rf_time)
check_quality(dt, "Одиночное дерево", dt_time)

--- Случайный лес ---
MAE: 818.29 | RMSE: 1072.45 | R2: 0.9138 | Время: 0.0544s
--- Одиночное дерево ---
MAE: 882.61 | RMSE: 1227.38 | R2: 0.8871 | Время: 0.0033s


 Обучение моделей
1. Дерево обучилось гораздо быстрее, добиться большей скорости для леса можно только уменьшая количество его деревьев или через параметр n_jobs=-1.
2. Лес обычно лучше по всем метрикам, так как он усредняет прогнозы и меньше переобучается.

In [20]:


for model, name in zip([rf, dt], ['Случайный лес', 'Одиночное дерево']):
    pred = model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, pred))
    r2 = r2_score(y_test, pred)
    print(f"{name}: RMSE = {rmse:.2f}, R2 = {r2:.4f}")

Случайный лес: RMSE = 1072.45, R2 = 0.9138
Одиночное дерево: RMSE = 1227.38, R2 = 0.8871


In [21]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import time

lr = LinearRegression()
t0 = time.time()
lr.fit(X_tr, y_tr)
lr_time = time.time() - t0

# 2. Функция для сбора метрик
def get_results(model, name, t):
    p = model.predict(X_te)
    return {
        'Model': name,
        'MAE': mean_absolute_error(y_te, p),
        'RMSE': np.sqrt(mean_squared_error(y_te, p)),
        'R2': r2_score(y_te, p),
        'Time (s)': t
    }

# Собираем данные в таблицу
results = [
    get_results(lr, 'Linear Regression', lr_time),
    get_results(dt, 'Decision Tree', dt_time),
    get_results(rf, 'Random Forest', rf_time)
]

df_res = pd.DataFrame(results).set_index('Model')
print(df_res.round(4))


                        MAE       RMSE      R2  Time (s)
Model                                                   
Linear Regression  952.6072  1432.0526  0.8463    0.0045
Decision Tree      882.6115  1227.3783  0.8871    0.0033
Random Forest      818.2940  1072.4483  0.9138    0.0544


Оценка качества и сравнение моделей
1. Использовал MAE, RMSE, R2. MAE так как она хорошо показывает среднее отклонение предсказания от реальности в тех же единицах, в которых измеряется целевая переменная. RMSE так как она более чувствительна к крупным отклонениям модели. Ну а R2 которая показывает качество модели в целом, в нашем случае R2 = 0,91 дает нам понять что модель почти идеальная, но есть мелкий шум.
2. Метрики считались на тестовой выборке.
3. Если судить по метрикам, то случайный лес справился лучше.
4. Как по мне, результат очень хороший.